# Phase 5 — Machine Learning : Détection d'Anomalies

**Bourse de Casablanca — Système de Surveillance Intelligent**

Ce notebook entraîne et évalue deux modèles non supervisés :

| Modèle | Principe | Avantage |
|--------|----------|----------|
| **Isolation Forest** | Partitionnement aléatoire — anomalies isolées rapidement | Rapide, robuste, interprétable |
| **Autoencoder** | Reconstruction — anomalies = erreur de reconstruction élevée | Capture les patterns complexes, non-linéaire |

Les deux modèles opèrent sur 3 niveaux :
- **Marché Global** : MASI rendements, volatilité, breadth, HHI
- **Instrument** : retours, volatilités, RSI, spread, turnover
- **Flux d'Ordres** : OIR, OAR, VWAP déviation, volatilité intraday

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR    = Path('../data')
REPORTS_DIR = Path('../reports')
MODELS_DIR  = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
BVC_BLUE  = '#002366'
BVC_GOLD  = '#C8A84B'

print('Imports OK')

## 1. Chargement des données (indicateurs Phase 2)

In [ ]:
df_market     = pd.read_parquet(DATA_DIR / 'market_indicators.parquet')
df_instrument = pd.read_parquet(DATA_DIR / 'instrument_indicators.parquet')
df_orderflow  = pd.read_parquet(DATA_DIR / 'orderflow_indicators.parquet')
df_stat_anom  = pd.read_parquet(DATA_DIR / 'all_anomalies.parquet')

print(f'Marché Global    : {df_market.shape[0]:>5} séances, {df_market.shape[1]} colonnes')
print(f'Instruments      : {df_instrument.shape[0]:>5} lignes,  {df_instrument.shape[1]} colonnes')
print(f'Flux d\'Ordres    : {df_orderflow.shape[0]:>5} lignes,  {df_orderflow.shape[1]} colonnes')
print(f'Anomalies stat.  : {df_stat_anom.shape[0]:>5} anomalies')
print()
print('Features disponibles — Marché :')
from src.ml_models import MARKET_FEATURES, INSTRUMENT_FEATURES, ORDERFLOW_FEATURES
print([f for f in MARKET_FEATURES if f in df_market.columns])
print('Features disponibles — Instrument :')
print([f for f in INSTRUMENT_FEATURES if f in df_instrument.columns])
print('Features disponibles — Flux d\'Ordres :')
print([f for f in ORDERFLOW_FEATURES if f in df_orderflow.columns])

## 2. Exploration des distributions (avant modélisation)

In [ ]:
from src.ml_models import MARKET_FEATURES, INSTRUMENT_FEATURES

mkt_feats  = [f for f in MARKET_FEATURES    if f in df_market.columns]
inst_feats = [f for f in INSTRUMENT_FEATURES if f in df_instrument.columns]

fig, axes = plt.subplots(2, len(mkt_feats), figsize=(4*len(mkt_feats), 6))
fig.suptitle('Distribution des Features — Marché Global', fontsize=14, color=BVC_BLUE, fontweight='bold')

for i, feat in enumerate(mkt_feats):
    s = df_market[feat].dropna()
    axes[0, i].hist(s, bins=30, color=BVC_BLUE, alpha=0.7, edgecolor='white')
    axes[0, i].set_title(feat, fontsize=8)
    axes[0, i].set_xlabel('')
    axes[1, i].boxplot(s, vert=True, patch_artist=True,
                        boxprops=dict(facecolor=BVC_GOLD, alpha=0.7))
    axes[1, i].set_title(f'Boxplot {feat}', fontsize=8)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'ml_01_distributions_marche.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée')

## 3. Entraînement Isolation Forest

In [ ]:
from src.ml_models import IsolationForestDetector

print('Entraînement Isolation Forest (contamination=5%)...')
if_detector = IsolationForestDetector(
    contamination=0.05,
    n_estimators=200,
    random_state=42,
)
if_detector.fit(df_market, df_instrument, df_orderflow)
print('✓ Modèles entraînés pour les niveaux :', list(if_detector.models.keys()))

# Sauvegarde
path = if_detector.save()
print(f'✓ Sauvegardé → {path}')

In [ ]:
# Application sur les données d'entraînement
from src.ml_models import apply_ml_detection

df_market_if     = apply_ml_detection(df_market,     'market',     if_detector)
df_instrument_if = apply_ml_detection(df_instrument, 'instrument', if_detector)
df_orderflow_if  = apply_ml_detection(df_orderflow,  'orderflow',  if_detector)

print('Résultats Isolation Forest — Marché :')
print(df_market_if[['IF_Score', 'IF_IsAnomaly', 'IF_Severity']].describe().round(3))

print(f"\nAnomalies IF — Marché    : {df_market_if['IF_IsAnomaly'].sum()} séances")
print(f"Anomalies IF — Instrument: {df_instrument_if['IF_IsAnomaly'].sum()} observations")
print(f"Anomalies IF — OrderFlow : {df_orderflow_if['IF_IsAnomaly'].sum()} observations")

In [ ]:
# Visualisation des scores IF — Marché
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Isolation Forest — Scores Marché Global', fontsize=13, color=BVC_BLUE, fontweight='bold')

# Score temporel
if 'Jour' in df_market_if.columns:
    dates = pd.to_datetime(df_market_if['Jour'])
    axes[0].plot(dates, df_market_if['IF_Score'], color=BVC_BLUE, alpha=0.6, linewidth=0.8, label='IF Score')
    anom = df_market_if[df_market_if['IF_IsAnomaly'] == 1]
    if len(anom) > 0 and 'Jour' in anom.columns:
        axes[0].scatter(pd.to_datetime(anom['Jour']), anom['IF_Score'],
                         color='red', s=60, zorder=5, label='Anomalies')
    axes[0].axhline(df_market_if['IF_Score'].quantile(0.95), color='orange', linestyle='--', alpha=0.8, label='P95')
    axes[0].set_title('Score IF dans le temps')
    axes[0].set_xlabel('Date')
    axes[0].set_ylabel('Score IF')
    axes[0].legend()
    axes[0].tick_params(axis='x', rotation=45)

# Distribution des sévérités
sev_counts = df_market_if['IF_Severity'].value_counts()
colors_sev = {'Normal': '#4CAF50', 'Faible': '#FFC107', 'Modéré': '#FF9800', 'Critique': '#F44336'}
c = [colors_sev.get(s, '#9E9E9E') for s in sev_counts.index]
axes[1].bar(sev_counts.index, sev_counts.values, color=c, edgecolor='white')
axes[1].set_title('Distribution des Sévérités IF')
axes[1].set_xlabel('Sévérité')
axes[1].set_ylabel('Nombre de séances')
for i, v in enumerate(sev_counts.values):
    axes[1].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'ml_02_if_scores_marche.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top anomalies IF — Instruments
if 'IF_IsAnomaly' in df_instrument_if.columns:
    top_if = df_instrument_if[df_instrument_if['IF_IsAnomaly'] == 1].copy()
    cols = [c for c in ['Jour', 'Ticker', 'IF_Score', 'IF_Severity', 
                         'Rendement_pct', 'Volatilite_10j', 'Volume_Relatif', 'RSI_14']
            if c in top_if.columns]
    print(f'Top 15 anomalies IF — Instruments (sur {len(top_if)} total) :')
    print(top_if[cols].sort_values('IF_Score', ascending=False).head(15).to_string(index=False))

## 4. Entraînement Autoencoder

In [ ]:
from src.ml_models import AutoencoderDetector

print('Entraînement Autoencoder (50 epochs, EarlyStopping)...')
ae_detector = AutoencoderDetector(
    encoding_dim=8,
    epochs=50,
    batch_size=32,
    contamination=0.05,
)
try:
    ae_detector.fit(df_market, df_instrument, df_orderflow, verbose=0)
    print('✓ Autoencoders entraînés pour :', list(ae_detector.models.keys()))
    path = ae_detector.save()
    print(f'✓ Sauvegardé → {path}')
    AE_OK = True
except ImportError as e:
    print(f'⚠ TensorFlow non disponible : {e}')
    print('  → Seul Isolation Forest sera utilisé.')
    AE_OK = False
    ae_detector = None

In [ ]:
if AE_OK and ae_detector is not None:
    df_market_ae     = apply_ml_detection(df_market,     'market',     ae_detector=ae_detector)
    df_instrument_ae = apply_ml_detection(df_instrument, 'instrument', ae_detector=ae_detector)

    print('Résultats Autoencoder — Marché :')
    print(df_market_ae[['AE_ReconError', 'AE_IsAnomaly', 'AE_Severity']].describe().round(4))
    print(f"\nAnomalies AE — Marché    : {df_market_ae['AE_IsAnomaly'].sum()} séances")
    print(f"Anomalies AE — Instrument: {df_instrument_ae['AE_IsAnomaly'].sum()} observations")

In [ ]:
if AE_OK and ae_detector is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Autoencoder — Erreurs de Reconstruction', fontsize=13, color=BVC_BLUE, fontweight='bold')

    # Erreurs reconstruction temporelles
    if 'Jour' in df_market_ae.columns:
        dates = pd.to_datetime(df_market_ae['Jour'])
        axes[0].plot(dates, df_market_ae['AE_ReconError'], color=BVC_BLUE, alpha=0.7, linewidth=0.9, label='Erreur AE')
        anom_ae = df_market_ae[df_market_ae['AE_IsAnomaly'] == 1]
        if len(anom_ae) > 0:
            axes[0].scatter(pd.to_datetime(anom_ae['Jour']), anom_ae['AE_ReconError'],
                             color='red', s=60, zorder=5, label='Anomalies')
        axes[0].axhline(ae_detector.thresholds.get('market', 0), color='orange',
                         linestyle='--', alpha=0.9, label='Seuil (P95)')
        axes[0].set_title('Erreur de reconstruction — Marché')
        axes[0].legend()
        axes[0].tick_params(axis='x', rotation=45)

    # Histogramme erreurs
    axes[1].hist(df_market_ae['AE_ReconError'].dropna(), bins=40, color=BVC_BLUE, alpha=0.7, edgecolor='white')
    axes[1].axvline(ae_detector.thresholds.get('market', 0), color='red', linestyle='--', linewidth=2, label='Seuil')
    axes[1].set_title('Distribution erreurs reconstruction')
    axes[1].set_xlabel('MSE Reconstruction')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'ml_03_ae_reconstruction_marche.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Score Combiné ML (IF + AE)

In [ ]:
from src.ml_models import apply_ml_detection, get_ml_anomalies

print('Calcul scores combinés IF + AE...')

df_market_ml = apply_ml_detection(
    df_market, 'market',
    if_detector=if_detector,
    ae_detector=ae_detector if AE_OK else None,
)
df_instrument_ml = apply_ml_detection(
    df_instrument, 'instrument',
    if_detector=if_detector,
    ae_detector=ae_detector if AE_OK else None,
)
df_orderflow_ml = apply_ml_detection(
    df_orderflow, 'orderflow',
    if_detector=if_detector,
    ae_detector=ae_detector if AE_OK else None,
)

# Synthèse
for name, df in [('Marché', df_market_ml), ('Instrument', df_instrument_ml), ('OrderFlow', df_orderflow_ml)]:
    n_anom = int(df['ML_IsAnomaly'].sum())
    n_crit = int((df['ML_Severity'] == 'Critique').sum())
    n_mod  = int((df['ML_Severity'] == 'Modéré').sum())
    print(f'{name:12s} → {n_anom:4d} anomalies ML  ({n_crit} Critique, {n_mod} Modéré)')

# Sauvegarde
df_market_ml.to_parquet(DATA_DIR / 'market_ml.parquet', index=False)
df_instrument_ml.to_parquet(DATA_DIR / 'instrument_ml.parquet', index=False)
df_orderflow_ml.to_parquet(DATA_DIR / 'orderflow_ml.parquet', index=False)
print('\n✓ Résultats ML sauvegardés en Parquet')

In [ ]:
# Top anomalies ML toutes classes
anom_market = get_ml_anomalies(df_market_ml, 'Marche')
anom_instr  = get_ml_anomalies(df_instrument_ml, 'Instrument', ticker_col='Ticker')
anom_of     = get_ml_anomalies(df_orderflow_ml, 'OrderFlow', ticker_col='Ticker')

all_ml_anom = pd.concat([anom_market, anom_instr, anom_of], ignore_index=True)
all_ml_anom.to_parquet(DATA_DIR / 'ml_anomalies.parquet', index=False)
all_ml_anom.to_csv(DATA_DIR / 'ml_anomalies.csv', index=False)

print(f'Total anomalies ML : {len(all_ml_anom)}')
print(all_ml_anom.groupby(['Niveau', 'ML_Severity'])['ML_Score'].count().to_string())

## 6. Comparaison Statistique vs ML

In [ ]:
from src.ml_models import compare_methods

# Comparaison au niveau instrument
stat_instr = df_stat_anom[df_stat_anom['Niveau'] == 'Instrument'].copy() if 'Niveau' in df_stat_anom.columns else df_stat_anom.copy()
cmp = compare_methods(stat_instr, anom_instr)

if not cmp.empty:
    print('Comparaison Statistique vs ML — Niveau Instrument :')
    print(f"  Anomalies stat.   : {cmp['Stat_Anomaly'].sum()}")
    print(f"  Anomalies ML      : {cmp['ML_Anomaly'].sum()}")
    print(f"  Consensus (les 2) : {cmp['Consensus'].sum()}")
    
    total = len(cmp)
    tp = cmp['Consensus'].sum()
    stat_only = ((cmp['Stat_Anomaly'] == 1) & (cmp['ML_Anomaly'] == 0)).sum()
    ml_only   = ((cmp['Stat_Anomaly'] == 0) & (cmp['ML_Anomaly'] == 1)).sum()
    print(f"  Stat. seul        : {stat_only} (non détectés par ML)")
    print(f"  ML seul           : {ml_only} (non détectés par stat.)")

In [ ]:
# Visualisation comparaison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Comparaison : Méthodes Statistiques vs Machine Learning', 
             fontsize=13, color=BVC_BLUE, fontweight='bold')

# Venn diagram simplifié
if not cmp.empty:
    stat_total = cmp['Stat_Anomaly'].sum()
    ml_total   = cmp['ML_Anomaly'].sum()
    consensus  = cmp['Consensus'].sum()
    stat_only_n = stat_total - consensus
    ml_only_n   = ml_total   - consensus

    categories = ['Stat. seul', 'Consensus', 'ML seul']
    values     = [stat_only_n, consensus, ml_only_n]
    colors     = [BVC_BLUE, BVC_GOLD, '#2E7D32']
    bars = axes[0].bar(categories, values, color=colors, edgecolor='white', width=0.5)
    axes[0].set_title('Répartition des anomalies\n(niveau Instrument)')
    axes[0].set_ylabel('Nombre')
    for bar in bars:
        h = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., h + 0.3, str(int(h)),
                     ha='center', fontweight='bold')

# Score ML distribution par niveau
if 'ML_Score' in df_instrument_ml.columns:
    axes[1].hist(df_instrument_ml['ML_Score'].dropna(), bins=50, 
                  color=BVC_BLUE, alpha=0.7, edgecolor='white')
    axes[1].axvline(80, color='red', linestyle='--', label='Seuil Critique (80)')
    axes[1].axvline(60, color='orange', linestyle='--', label='Seuil Modéré (60)')
    axes[1].set_title('Distribution ML Score — Instruments')
    axes[1].set_xlabel('ML Score (0-100)')
    axes[1].legend(fontsize=8)

# Top tickers par score ML
if 'Ticker' in df_instrument_ml.columns and 'ML_Score' in df_instrument_ml.columns:
    top_tickers = (df_instrument_ml.groupby('Ticker')['ML_Score']
                   .mean().sort_values(ascending=False).head(15))
    axes[2].barh(top_tickers.index, top_tickers.values, color=BVC_GOLD, edgecolor='white')
    axes[2].set_title('Top 15 Tickers — Score ML Moyen')
    axes[2].set_xlabel('Score ML Moyen')
    axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'ml_04_comparaison_stat_ml.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap anomalies ML par ticker / mois
if 'Jour' in df_instrument_ml.columns and 'Ticker' in df_instrument_ml.columns:
    df_instrument_ml['Mois'] = pd.to_datetime(df_instrument_ml['Jour']).dt.to_period('M').astype(str)
    pivot = (df_instrument_ml.groupby(['Ticker', 'Mois'])['ML_IsAnomaly']
              .sum().unstack(fill_value=0))
    top20 = pivot.sum(axis=1).sort_values(ascending=False).head(20).index
    pivot_top = pivot.loc[top20]

    fig, ax = plt.subplots(figsize=(14, 7))
    sns.heatmap(pivot_top, cmap='YlOrRd', annot=True, fmt='d', ax=ax,
                linewidths=0.3, cbar_kws={'label': 'Nb anomalies ML'})
    ax.set_title('Heatmap Anomalies ML — Top 20 Instruments par Mois',
                  fontsize=13, color=BVC_BLUE, fontweight='bold')
    ax.set_xlabel('Mois')
    ax.set_ylabel('Ticker')
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / 'ml_05_heatmap_ml_anomalies.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Synthèse finale

In [ ]:
print('=' * 60)
print('SYNTHÈSE — PHASE 5 : MACHINE LEARNING')
print('=' * 60)
print()
print('Modèles entraînés :')
print(f'  Isolation Forest : {list(if_detector.models.keys())}')
if AE_OK and ae_detector:
    print(f'  Autoencoder      : {list(ae_detector.models.keys())}')
    print(f'  Seuils AE        : {ae_detector.thresholds}')
else:
    print('  Autoencoder      : Non disponible (TF requis)')
print()
print('Anomalies détectées :')
if 'ML_IsAnomaly' in df_market_ml.columns:
    print(f'  Marché    : {int(df_market_ml["ML_IsAnomaly"].sum()):4d} / {len(df_market_ml):4d} séances')
if 'ML_IsAnomaly' in df_instrument_ml.columns:
    print(f'  Instrument: {int(df_instrument_ml["ML_IsAnomaly"].sum()):4d} / {len(df_instrument_ml):4d} obs.')
if 'ML_IsAnomaly' in df_orderflow_ml.columns:
    print(f'  OrderFlow : {int(df_orderflow_ml["ML_IsAnomaly"].sum()):4d} / {len(df_orderflow_ml):4d} obs.')
print()
print('Fichiers générés :')
for f in ['market_ml.parquet', 'instrument_ml.parquet', 'orderflow_ml.parquet', 
           'ml_anomalies.parquet', 'ml_anomalies.csv']:
    p = DATA_DIR / f
    size = p.stat().st_size / 1024 if p.exists() else 0
    print(f'  {f:<30s} {size:6.1f} KB')
print()
print('Phase 5 terminée ✓')
print('→ Prochain : Page Streamlit 5_Machine_Learning.py')